## Q: What is "Contextual Retrieval" (the Anthropic pattern)?
Strong answer: Contextual Retrieval involves prepending a 1-sentence global context to every chunk before embedding it. For example, if a chunk is about "battery life," but it's from a manual for a "2025 Model X Drone," the text [Drone_Model_X_Manual]: is added to the chunk. This ensures that the vector for "battery life" is influenced by the "Drone" context, preventing it from being accidentally retrieved for "phone battery" queries.



In [ ]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)
from openai import OpenAI
import uuid


# ============================================
# CONFIG
# ============================================

DOCUMENT_PATH = "documents/tesla_manual.txt"

COLLECTION_NAME = "contextual_rag"

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

# NVIDIA NIM
client = OpenAI(
    base_url="YOUR_NVIDIA_NIM_ENDPOINT/v1",
    api_key="YOUR_API_KEY"
)

# ============================================
# LOAD DOCUMENT
# ============================================

document = Path(DOCUMENT_PATH).read_text()

# ============================================
# CHUNK DOCUMENT
# ============================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_text(document)

print(f"Total Chunks: {len(chunks)}")

# ============================================
# EMBEDDING MODEL
# ============================================

embedder = SentenceTransformer(
    EMBEDDING_MODEL
)

# ============================================
# CONTEXT GENERATION FUNCTION
# ============================================

def generate_context(full_document, chunk):

    prompt = f"""
You are creating contextual retrieval metadata.

Given the FULL DOCUMENT and a CHUNK from it,
write ONE concise sentence describing
what this chunk is about in the context
of the entire document.

FULL DOCUMENT:
{full_document[:12000]}

CHUNK:
{chunk}

Return only the context sentence.
"""

    response = client.chat.completions.create(
        model="meta/llama-3.1-70b-instruct",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()


# ============================================
# PREPARE CONTEXTUAL CHUNKS
# ============================================

processed_chunks = []

for idx, chunk in enumerate(chunks):

    print(f"Processing Chunk {idx+1}/{len(chunks)}")

    context = generate_context(
        document,
        chunk
    )

    contextual_text = f"""
Context:
{context}

Content:
{chunk}
"""

    embedding = embedder.encode(
        contextual_text,
        normalize_embeddings=True
    ).tolist()

    processed_chunks.append(
        {
            "id": str(uuid.uuid4()),
            "chunk": chunk,
            "context": context,
            "contextual_text": contextual_text,
            "embedding": embedding
        }
    )

print("All chunks contextualized.")